In [53]:
from pathlib import Path
import pandas as pd

def get_project_root():
    return Path().resolve().parent


df = pd.read_csv(get_project_root().parent / "HH+error_analysis_v2.csv", delimiter=";",dtype={"indicator_id": str})
df_v3 = pd.read_csv(get_project_root().parent / "error_analysis_v3.csv",dtype={"indicator_id": str})
df_v4 = pd.read_csv(get_project_root().parent / "error_analysis_v4.csv",dtype={"indicator_id": str})


In [54]:
df[['fullname', 'filename', 'indicator_id', 'indicator_value', 'dimension',
       'golden_label', 'pred_label', 'value', 'pred_value', 'error_type',
       'evidence', 'rationale', 'golden_label_update', 'explanation']]

,fullname,filename,indicator_id,indicator_value,dimension,golden_label,pred_label,value,pred_value,error_type,evidence,rationale,golden_label_update,explanation
0,ACEH BIREUEN,ACEH_BIREUEN.txt,1.9,Sexual slavery / exploitation,scope_of_violence,Yes,No,1.0,0.0,false_negative,NaN,While exploitation is mentioned in employment ...,x,"Chapter 1, Article 1, no. 14 mentions special ..."
1,ACEH BIREUEN,ACEH_BIREUEN.txt,3.7,Psychosocial counselling,specialised_support_services,Yes,No,1.0,0.0,false_negative,NaN,"Although the policy references rehabilitation,...",x,"Paragraph 8, article 19, no.2 mentions psychol..."
2,ACEH BIREUEN,ACEH_BIREUEN.txt,4.2,Gender equality education,primary_prevention,No,Yes,0.0,1.0,false_positive,Article 5\n(1) Every woman has the right to:\n...,"The policy explicitly sets out women’s rights,...",x,NaN
3,ACEH BIREUEN,ACEH_BIREUEN.txt,4.4,Prevention through education,primary_prevention,No,Yes,0.0,1.0,false_positive,Article 25\n(1) The Regency Government is obli...,The policy includes prevention efforts through...,Yes,NaN
4,ACEH BIREUEN,ACEH_BIREUEN.txt,5.1,NGO collaboration,stakeholder_engagement,No,Yes,0.0,1.0,false_positive,CHAPTER VIII\nCOMMUNITY PARTICIPATION\nArticle...,The policy provides for community participatio...,Yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1084,SUMATERA UTARA PADANGSIDIMPUAN KOTA,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,6.3,Women workers,intersectional_approach,No,Yes,0.0,1.0,false_positive,Article 5\n(1) Every woman has the right to:\n...,The policy explicitly mentions work-related pr...,NaN,NaN
1085,SUMATERA UTARA PAKPAK BHARAT,SUMATERA_UTARA_PAKPAK_BHARAT.txt,1.9,Sexual slavery / exploitation,scope_of_violence,No,Yes,0.0,1.0,false_positive,Article 3\n(1) The objectives of providing pro...,The policy states an objective to eliminate al...,NaN,NaN
1086,SUMATERA UTARA PAKPAK BHARAT,SUMATERA_UTARA_PAKPAK_BHARAT.txt,3.1,Economic support services,specialised_support_services,Yes,No,1.0,0.0,false_negative,NaN,While prevention efforts mention access to tra...,NaN,NaN
1087,SUMATERA UTARA PAKPAK BHARAT,SUMATERA_UTARA_PAKPAK_BHARAT.txt,4.2,Gender equality education,primary_prevention,No,Yes,0.0,1.0,false_positive,CHAPTER XII\nCOMMUNITY PARTICIPATION\nArticle ...,The policy explicitly mentions community-led s...,NaN,NaN


In [55]:
import pandas as pd

# Columns that uniquely identify an example
merge_keys = [
    "fullname",
    "filename",
    "indicator_id",
]

# ------------------------------------------------------------------
# 1. Bring human review columns onto the new model output
# ------------------------------------------------------------------

human_cols = merge_keys + [
    "golden_label_update",
    "explanation",
]

df_v4 = df_v4.merge(
    df[human_cols],
    on=merge_keys,
    how="left",
)

# ------------------------------------------------------------------
# 2. Compare old vs new errors
# ------------------------------------------------------------------

old_errors = set(map(tuple, df[merge_keys].to_numpy()))
new_errors = set(map(tuple, df_v4[merge_keys].to_numpy()))

corrected = old_errors - new_errors      # error disappeared
new = new_errors - old_errors            # brand new error
persisting = old_errors & new_errors     # still an error

print(f"Old errors:        {len(old_errors)}")
print(f"New model errors:  {len(new_errors)}")
print(f"Corrected errors:  {len(corrected)}")
print(f"Persisting errors: {len(persisting)}")
print(f"New errors:        {len(new)}")

# ------------------------------------------------------------------
# 3. Create a summary dataframe
# ------------------------------------------------------------------

summary = pd.DataFrame({
    "Metric": [
        "Old errors",
        "New model errors",
        "Corrected by new model",
        "Persisting errors",
        "New errors introduced",
    ],
    "Count": [
        len(old_errors),
        len(new_errors),
        len(corrected),
        len(persisting),
        len(new),
    ]
})

print(summary)

# ------------------------------------------------------------------
# 4. (Optional) Label each row in df_v4
# ------------------------------------------------------------------

df_v4["_key"] = list(map(tuple, df_v4[merge_keys].to_numpy()))

df_v4["error_status"] = df_v4["_key"].map(
    lambda k: "Persisting error" if k in persisting else "New error"
)

df_v4.drop(columns="_key", inplace=True)

# ------------------------------------------------------------------
# 5. Save
# ------------------------------------------------------------------

# df_v4.to_csv("df_v4_with_human_review.csv", index=False)
# summary.to_csv("error_summary.csv", index=False)

Old errors:        1081
New model errors:  928
Corrected errors:  556
Persisting errors: 525
New errors:        403
                   Metric  Count
0              Old errors   1081
1        New model errors    928
2  Corrected by new model    556
3       Persisting errors    525
4   New errors introduced    403


In [56]:
keys = ['filename', 'indicator_id','indicator_value', 'error_type']

intersection = (
    df[keys]
    .merge(df_v3[keys], on=keys, how='inner')
    .merge(df_v4[keys], on=keys, how='inner')
    .drop_duplicates()
)

intersection

,filename,indicator_id,indicator_value,error_type
0,ACEH_BIREUEN.txt,5.4,Education institutions engagement,false_positive
1,ACEH_BIREUEN.txt,7.3,International donors,false_positive
2,ACEH_SUBULUSSALAM_KOTA.txt,5.2,Religious engagement,false_negative
3,ACEH_SUBULUSSALAM_KOTA.txt,6.3,Women workers,false_positive
4,BALI_BADUNG.txt,2.5,Training and capacity building,false_positive
...,...,...,...,...
302,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,4.2,Gender equality education,false_positive
303,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,4.4,Prevention through education,false_positive
304,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,5.8,Private sector engagement,false_positive
305,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,6.3,Women workers,false_positive


In [57]:
intersection_with_update = intersection.merge(
    df[keys + ['golden_label_update']].drop_duplicates(subset=keys),
    on=keys,
    how='left'
)

In [58]:
a = intersection_with_update[
    ['filename', 'indicator_id','indicator_value', 'error_type', 'golden_label_update']
]#.dropna()
a

,filename,indicator_id,indicator_value,error_type,golden_label_update
0,ACEH_BIREUEN.txt,5.4,Education institutions engagement,false_positive,x
1,ACEH_BIREUEN.txt,7.3,International donors,false_positive,x
2,ACEH_SUBULUSSALAM_KOTA.txt,5.2,Religious engagement,false_negative,x
3,ACEH_SUBULUSSALAM_KOTA.txt,6.3,Women workers,false_positive,x
4,BALI_BADUNG.txt,2.5,Training and capacity building,false_positive,x
...,...,...,...,...,...
293,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,4.2,Gender equality education,false_positive,NaN
294,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,4.4,Prevention through education,false_positive,NaN
295,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,5.8,Private sector engagement,false_positive,NaN
296,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,6.3,Women workers,false_positive,NaN


In [59]:
a = intersection_with_update[
    ['filename','indicator_id', 'indicator_value', 'error_type', 'golden_label_update']
].merge(
    df_v4[keys + ['value', 'pred_value', 'evidence', 'rationale']],
    on=keys,
    how='left'
)

a

,filename,indicator_id,indicator_value,error_type,golden_label_update,value,pred_value,evidence,rationale
0,ACEH_BIREUEN.txt,5.4,Education institutions engagement,false_positive,x,0.0,1.0,"c. improve coordination, integration, synchron...",The policy explicitly includes academics among...
1,ACEH_BIREUEN.txt,7.3,International donors,false_positive,x,0.0,1.0,f. coordinating the participation of the busin...,The policy explicitly references coordination ...
2,ACEH_SUBULUSSALAM_KOTA.txt,5.2,Religious engagement,false_negative,x,1.0,0.0,NaN,While the policy references Islamic Sharia and...
3,ACEH_SUBULUSSALAM_KOTA.txt,6.3,Women workers,false_positive,x,0.0,1.0,d. Women obtain the right to special protectio...,The policy explicitly recognizes a right to sp...
4,BALI_BADUNG.txt,2.5,Training and capacity building,false_positive,x,0.0,1.0,a. coverage of Women and Child Victims of Viol...,The policy’s minimum service standards include...
...,...,...,...,...,...,...,...,...,...
298,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,4.2,Gender equality education,false_positive,NaN,0.0,1.0,(1) Gender Mainstreaming or abbreviated as PUG...,The policy explicitly frames gender mainstream...
299,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,4.4,Prevention through education,false_positive,NaN,0.0,1.0,(1) Prevention of acts of violence against wom...,The policy explicitly states that prevention o...
300,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,5.8,Private sector engagement,false_positive,NaN,0.0,1.0,To guarantee the fulfillment of children's rig...,The policy explicitly includes the business se...
301,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,6.3,Women workers,false_positive,NaN,0.0,1.0,d. obtain decent work in accordance with her c...,The policy explicitly addresses women in the c...


In [60]:
output_path = get_project_root().parent / "error_analysis_v4_to_review.csv"
a.to_csv(output_path, index=False)